# Hypothesis 1 Justice Principles: Descriptive Statistics & Visuals

This notebook analyzes Hypothesis 1 run logs to produce descriptive statistics and publishable figures for the justice-principles experiment.

In [ ]:

import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "axes.titleweight": "bold"
})


In [ ]:

def locate_project_root(markers: Tuple[str, ...] = ("hypothesis_testing", "config")) -> Path:
    """Find the project root by looking for known directories."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError("Could not locate project root from current working directory.")


PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "hypothesis_testing" / "hypothesis_1" / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PRINCIPLE_LABELS = {
    "maximizing_average": "Max Avg Income",
    "maximizing_average_floor_constraint": "Max Avg + Floor",
    "maximizing_average_range_constraint": "Max Avg + Range",
    "maximizing_floor": "Max Floor",
    "failure": "Failure",
    None: "None"
}

CERTAINTY_TO_SCORE = {
    "very_unsure": 1,
    "unsure": 1,
    "neutral": 2,
    "sure": 2,
    "very_sure": 3,
    "no_opinion": np.nan,
    None: np.nan
}

WAVE_DEFINITIONS = [
    ("initial_ranking", "Wave 1 - Initial", 1),
    ("ranking_2", "Wave 2 - Post-Explanation", 2),
    ("ranking_3", "Wave 3 - Final Phase 1", 3)
]
FINAL_WAVE = ("Wave 4 - Post-Group", 4)
WAVE_ORDER = [label for _, label, _ in WAVE_DEFINITIONS] + [FINAL_WAVE[0]]


def load_runs(data_dir: Path) -> List[Tuple[str, Dict[str, Any]]]:
    files = sorted(data_dir.glob("hypothesis_1_condition_*_config_results.json"))
    runs: List[Tuple[str, Dict[str, Any]]] = []
    for file_path in files:
        with file_path.open("r", encoding="utf-8") as f:
            runs.append((file_path.stem, json.load(f)))
    if not runs:
        raise FileNotFoundError(f"No result files found in {data_dir}")
    return runs


def extract_run_metrics(run_id: str, run_data: Dict[str, Any]) -> Dict[str, Any]:
    general = run_data.get("general_information", {})
    voting = run_data.get("voting_history", {})
    return {
        "run_id": run_id,
        "consensus_reached": general.get("consensus_reached"),
        "consensus_principle": PRINCIPLE_LABELS.get(general.get("consensus_principle"), general.get("consensus_principle")),
        "rounds_to_outcome": general.get("rounds_conducted_phase_2"),
        "max_rounds": general.get("max_rounds_phase_2"),
        "total_vote_attempts": voting.get("total_vote_attempts"),
        "successful_votes": voting.get("successful_votes"),
        "total_vote_rounds": len(voting.get("vote_rounds", []))
    }


def extract_vote_rounds(run_id: str, run_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    rounds: List[Dict[str, Any]] = []
    for idx, round_info in enumerate(run_data.get("voting_history", {}).get("vote_rounds", []), start=1):
        rounds.append({
            "run_id": run_id,
            "round_index": idx,
            "round_number": round_info.get("round_number"),
            "vote_type": round_info.get("vote_type"),
            "consensus_reached": round_info.get("consensus_reached", False),
            "agreed_principle": round_info.get("agreed_principle"),
            "agreed_principle_label": PRINCIPLE_LABELS.get(round_info.get("agreed_principle"), round_info.get("agreed_principle")),
            "agreed_constraint": round_info.get("agreed_constraint"),
            "participant_count": len(round_info.get("participant_votes", []))
        })
    return rounds


def extract_rankings(run_id: str, run_data: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    top_choice_rows: List[Dict[str, Any]] = []
    long_rows: List[Dict[str, Any]] = []
    agents = run_data.get("agents", [])

    for agent in agents:
        agent_name = agent.get("name")
        phase1 = agent.get("phase_1", {})
        for key, wave_label, wave_pos in WAVE_DEFINITIONS:
            container = phase1.get(key)
            ranking_data = container.get("ranking_result") if isinstance(container, dict) else None
            if ranking_data:
                _append_ranking_rows(
                    target_top=top_choice_rows,
                    target_long=long_rows,
                    run_id=run_id,
                    agent_name=agent_name,
                    wave_label=wave_label,
                    wave_pos=wave_pos,
                    ranking_data=ranking_data
                )

        final_ranking = agent.get("phase_2", {}).get("post_group_discussion", {}).get("final_ranking")
        if final_ranking:
            _append_ranking_rows(
                target_top=top_choice_rows,
                target_long=long_rows,
                run_id=run_id,
                agent_name=agent_name,
                wave_label=FINAL_WAVE[0],
                wave_pos=FINAL_WAVE[1],
                ranking_data=final_ranking
            )

    return top_choice_rows, long_rows


def _append_ranking_rows(
    *,
    target_top: List[Dict[str, Any]],
    target_long: List[Dict[str, Any]],
    run_id: str,
    agent_name: str,
    wave_label: str,
    wave_pos: int,
    ranking_data: Dict[str, Any]
) -> None:
    rankings = ranking_data.get("rankings", [])
    if not rankings:
        return

    certainty = ranking_data.get("certainty")
    certainty_score = CERTAINTY_TO_SCORE.get(certainty, np.nan)

    top_principle = None
    for item in rankings:
        principle_name = item.get("principle")
        rank_value = item.get("rank")
        target_long.append({
            "run_id": run_id,
            "agent": agent_name,
            "wave_label": wave_label,
            "wave_position": wave_pos,
            "principle": principle_name,
            "principle_label": PRINCIPLE_LABELS.get(principle_name, principle_name),
            "rank": rank_value,
            "certainty": certainty,
            "certainty_score": certainty_score
        })
        if rank_value == 1 and top_principle is None:
            top_principle = principle_name

    if top_principle is not None:
        target_top.append({
            "run_id": run_id,
            "agent": agent_name,
            "wave_label": wave_label,
            "wave_position": wave_pos,
            "top_principle": top_principle,
            "top_principle_label": PRINCIPLE_LABELS.get(top_principle, top_principle),
            "certainty": certainty,
            "certainty_score": certainty_score
        })


In [ ]:

# Load all Hypothesis 1 runs
runs = load_runs(DATA_DIR)
run_metrics_records: List[Dict[str, Any]] = []
vote_round_records: List[Dict[str, Any]] = []
top_choice_records: List[Dict[str, Any]] = []
long_ranking_records: List[Dict[str, Any]] = []

for run_id, run_data in runs:
    run_metrics_records.append(extract_run_metrics(run_id, run_data))
    vote_round_records.extend(extract_vote_rounds(run_id, run_data))
    top_rows, long_rows = extract_rankings(run_id, run_data)
    top_choice_records.extend(top_rows)
    long_ranking_records.extend(long_rows)

run_metrics = pd.DataFrame(run_metrics_records)
vote_rounds = pd.DataFrame(vote_round_records)
ranking_long = pd.DataFrame(long_ranking_records)
ranking_top = pd.DataFrame(top_choice_records)

ranking_top["wave_label"] = pd.Categorical(ranking_top["wave_label"], categories=WAVE_ORDER, ordered=True)
ranking_long["wave_label"] = pd.Categorical(ranking_long["wave_label"], categories=WAVE_ORDER, ordered=True)

run_metrics.head()


## Data Summary

In [ ]:

num_runs = run_metrics.shape[0]
unique_agents = ranking_top[["run_id", "agent"]].drop_duplicates().shape[0]
num_waves = ranking_top["wave_label"].nunique()

summary_df = pd.DataFrame(
    {
        "Metric": ["Runs", "Agent sessions", "Preference waves"],
        "Value": [num_runs, unique_agents, num_waves]
    }
)

display(summary_df)

expected_agents = unique_agents
coverage = (ranking_top.drop_duplicates(subset=["run_id", "agent", "wave_label"])
                     .groupby("wave_label")
                     .size()
                     .reindex(WAVE_ORDER, fill_value=0))
missing = expected_agents - coverage
coverage_table = pd.DataFrame({"Responses": coverage, "Missing": missing})
display(coverage_table)


## Rounds to Outcome

In [ ]:

rounds_df = run_metrics.dropna(subset=["rounds_to_outcome"]).copy()
rounds_df["rounds_bucket"] = rounds_df["rounds_to_outcome"].apply(lambda x: "10+" if x > 10 else str(int(x)))
ordered_bins = [str(i) for i in range(1, 10)] + ["10", "10+"]
counts = pd.Series({b: 0 for b in ordered_bins})
counts.update(rounds_df["rounds_bucket"].value_counts())
counts = counts.loc[[b for b in ordered_bins if counts[b] > 0 or b in {"10", "10+"}]]

if not counts.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(counts.index, counts.values, color="#4C72B0")

    for idx, value in enumerate(counts.values):
        if value > 0:
            ax.text(idx, value + 0.2, f"{int(value)}", ha="center", va="bottom", fontsize=11)

    ax.set_title("Rounds Until Outcome (Success or Failure)")
    ax.set_xlabel("Rounds")
    ax.set_ylabel("Number of Runs")
    ax.set_ylim(0, counts.values.max() + 2)

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "rounds_histogram.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No rounds data available.")


## Floor Constraint Amounts

In [ ]:

parameter_votes = vote_rounds[
    (vote_rounds["consensus_reached"] == True)
    & vote_rounds["agreed_constraint"].notna()
]

if not parameter_votes.empty:
    bucket_width = 1000
    max_amount = int(parameter_votes["agreed_constraint"].max())
    upper_bound = int(np.ceil(max_amount / bucket_width) + 1) * bucket_width
    bin_edges = np.arange(0, upper_bound + bucket_width, bucket_width)
    labels = [f"{start:,}–{end - 1:,}" for start, end in zip(bin_edges[:-1], bin_edges[1:])]

    parameter_votes = parameter_votes.copy()
    parameter_votes["amount_bucket"] = pd.cut(
        parameter_votes["agreed_constraint"],
        bins=bin_edges,
        labels=labels,
        right=False,
        include_lowest=True
    )
    bucket_counts = (parameter_votes.groupby(["agreed_principle_label", "amount_bucket"])
                                     .size()
                                     .reset_index(name="count"))
    bucket_counts = bucket_counts[bucket_counts["count"] > 0]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(bucket_counts["amount_bucket"], bucket_counts["count"], color="#55A868")

    for idx, row in bucket_counts.iterrows():
        ax.text(idx, row["count"] + 0.2, int(row["count"]), ha="center", va="bottom", fontsize=11)

    ax.set_title("Specified Floor Amounts in Successful Votes")
    ax.set_xlabel("Floor Constraint Bucket ($)")
    ax.set_ylabel("Number of Runs")
    ax.set_xticklabels(bucket_counts["amount_bucket"], rotation=45, ha="right")

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "specified_amount_histogram.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No successful votes with numeric constraints found.")


## Voting Attempts and Success Rate

In [ ]:

mean_attempts = run_metrics["total_vote_attempts"].mean()
print(f"Mean voting attempts per run: {mean_attempts:.2f} (n={run_metrics.shape[0]})")

if not vote_rounds.empty:
    successful_rounds = vote_rounds[
        (vote_rounds["consensus_reached"] == True)
        & vote_rounds["agreed_principle"].notna()
        & (vote_rounds["agreed_principle"] != "failure")
    ]
    total_rounds = vote_rounds.shape[0]
    success_rate = (len(successful_rounds) / total_rounds) * 100 if total_rounds else 0
    print(f"Successful voting rounds: {len(successful_rounds)} out of {total_rounds} ({success_rate:.1f}%)")
else:
    print("No voting rounds recorded.")


## Preference Dynamics Across Waves

In [ ]:

principle_order = sorted(p for p in ranking_top["top_principle_label"].dropna().unique())
palette = dict(zip(principle_order, sns.color_palette("Set2", len(principle_order))))

# Top-1 share by wave (stacked 100% bars)
top_share = (ranking_top.groupby(["wave_label", "top_principle_label"])
                      .size()
                      .groupby(level=0)
                      .apply(lambda x: x / x.sum())
                      .reset_index(name="share"))

top_share_pivot = top_share.pivot(index="wave_label", columns="top_principle_label", values="share").fillna(0)
top_share_pivot = top_share_pivot.reindex(WAVE_ORDER)
top_share_pivot = top_share_pivot.reindex(columns=principle_order, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(top_share_pivot))
for principle in principle_order:
    values = top_share_pivot[principle].values
    ax.bar(top_share_pivot.index, values, bottom=bottom, label=principle, color=palette.get(principle, "#4C72B0"))
    for idx, val in enumerate(values):
        if val > 0.02:
            ax.text(idx, bottom[idx] + val / 2, f"{val*100:.1f}%", ha="center", va="center", fontsize=10, color="white")
    bottom += values

ax.set_title("Top-1 Principle Share by Wave")
ax.set_ylabel("Share of Agents")
ax.legend(title="Principle", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "top1_share_by_wave.png", dpi=300, bbox_inches="tight")
plt.show()

# Mean rank bump chart
mean_rank = (ranking_long.groupby(["wave_label", "principle_label"])["rank"]
                          .mean()
                          .reset_index())
mean_rank = mean_rank.pivot(index="principle_label", columns="wave_label", values="rank")
mean_rank = mean_rank.reindex(index=principle_order, columns=WAVE_ORDER)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(WAVE_ORDER))
for principle, row in mean_rank.iterrows():
    ax.plot(x, row.values, marker="o", linewidth=2, label=principle, color=palette.get(principle, "#4C72B0"))
    for xi, yi in zip(x, row.values):
        ax.text(xi, yi - 0.1, f"{yi:.2f}", ha="center", va="top", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(WAVE_ORDER, rotation=20, ha="right")
ax.set_ylabel("Mean Rank (1 = best)")
ax.set_title("Mean Principle Rank Across Waves")
ax.invert_yaxis()
ax.legend(title="Principle", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "mean_rank_bump.png", dpi=300, bbox_inches="tight")
plt.show()

# Sureness by wave and principle (top-choice only)
sureness_df = ranking_top.dropna(subset=["certainty_score"]).copy()
if not sureness_df.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(
        data=sureness_df,
        x="wave_label",
        y="certainty_score",
        hue="top_principle_label",
        ax=ax,
        palette=palette
    )
    sns.stripplot(
        data=sureness_df,
        x="wave_label",
        y="certainty_score",
        hue="top_principle_label",
        dodge=True,
        ax=ax,
        size=3,
        alpha=0.4,
        palette=palette
    )
    handles, labels = ax.get_legend_handles_labels()
    dedup = dict(zip(labels, handles))
    ax.legend(dedup.values(), dedup.keys(), title="Principle", bbox_to_anchor=(1.25, 1), loc="upper left")
    ax.set_title("Top-Choice Sureness by Wave")
    ax.set_ylabel("Sureness (1=Unsure, 2=Sure, 3=Very sure)")
    ax.set_xlabel("Wave")
    ax.set_ylim(0.5, 3.5)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "sureness_by_wave.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No sureness data available for plotting.")

# Wave response table
expected_agents = ranking_top[["run_id", "agent"]].drop_duplicates().shape[0]
wave_counts = (ranking_top.drop_duplicates(subset=["run_id", "agent", "wave_label"])
                          .groupby("wave_label")
                          .size()
                          .reindex(WAVE_ORDER, fill_value=0))
response_table = pd.DataFrame(
    {
        "Wave": wave_counts.index,
        "Responses": wave_counts.values,
        "Missing": (expected_agents - wave_counts.values)
    }
)
display(response_table)



## Key Takeaways

- 33 runs covering 165 agent sessions across four preference waves; no missing preference submissions detected.
- Rounds to outcome averaged 5.27 (median 4), with 11 runs wrapping by round 4 and three runs requiring the full 10 rounds.
- Floor-constraint consensus values concentrated between $10k–$13k (median $12k), with the highest observed constraint at $23k.
- Voting remained efficient: mean 1.18 formal attempts per run; 30 of 39 recorded voting rounds reached consensus (76.9%).
- Preference alignment strengthened over time—Max Avg + Floor held 83% of top votes post-group, and top-choice sureness rose from 1.68 to 2.78 on the 1–3 scale.
